# EfficientNetV2-B3 — synthetic → original (transfer learning)

Trains **Classifier-Aug-Mask**: pre-train 6 epochs on synthetic masked faces, then fine-tune 3 epochs on the original Face Mask 12k data. This is the project's synthetic-data contribution.

CLI equivalent: two `python -m src.train` runs chained with `--init-weights`.

In [ ]:
from pathlib import Path

import keras
from keras.applications import EfficientNetV2B3
from keras import Sequential
from keras.layers import Flatten, Dense
from keras.preprocessing.image import ImageDataGenerator

import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
keras.utils.set_random_seed(0)
tf.random.set_seed(0)
np.random.seed(0)

In [ ]:
train_dir = '/kaggle/input/face-mask-synthetic/for_vgg/train'
test_dir = '/kaggle/input/face-mask-synthetic/for_vgg/test'
val_dir = '/kaggle/input/face-mask-synthetic/for_vgg/validation'

In [ ]:
batch_size = 128
input_dim = 64


def preprocess(x):
    blur_kernel = np.random.randint(0, 4) * 2 + 1
    x = cv2.GaussianBlur(x, (blur_kernel, blur_kernel), 0)
    return x


# Rescaling is done with include_preprocessing=True parameter of EfficientNetV2M

train_datagen = ImageDataGenerator(horizontal_flip=True, zoom_range=0.2, shear_range=0.2, brightness_range=[0.6, 1.0], preprocessing_function=preprocess)
train_generator = train_datagen.flow_from_directory(directory=train_dir,
                                                    target_size=(input_dim, input_dim),
                                                    class_mode='categorical',
                                                    batch_size=batch_size)

val_datagen = ImageDataGenerator()
val_generator = train_datagen.flow_from_directory(directory=val_dir,
                                                  target_size=(input_dim, input_dim),
                                                  class_mode='categorical',
                                                  batch_size=batch_size)

test_datagen = ImageDataGenerator()
test_generator = train_datagen.flow_from_directory(directory=val_dir,
                                                   target_size=(input_dim, input_dim),
                                                   class_mode='categorical',
                                                   batch_size=batch_size)

In [ ]:
# fig, ax = plt.subplots(2, 3, figsize=(9, 7))
# ax = ax.flatten()

# img_batch = train_generator[0][0]
# label_batch = train_generator[0][1]
# for i, label, img in zip(np.arange(6), label_batch, img_batch):
#     ax[i].set_title(label)
#     ax[i].imshow(img / 255)
#     ax[i].axis('off')

# plt.show()

In [ ]:
efficient_net = EfficientNetV2B3(weights='imagenet', include_top=False, input_shape=(input_dim, input_dim, 3), include_preprocessing=True)

for layer in efficient_net.layers:
    layer.trainable = False
    
model = Sequential()
model.add(efficient_net)
model.add(Flatten())
model.add(Dense(2, activation='softmax'))
model.summary()

In [ ]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
history = model.fit(train_generator,
                    steps_per_epoch=len(train_generator),
                    epochs=6,
                    validation_data=val_generator,
                    validation_steps=len(val_generator))

In [ ]:
model.evaluate(test_generator)

In [ ]:
model.save('efficientnetv2_b3_syn6.h5')

In [ ]:
import json

with open('history_syn6.json', 'w') as f:
    json.dump(history.history, f, indent=4)

In [ ]:
train_dir = '/kaggle/input/face-mask-12k-images-dataset/Face Mask Dataset/Train'
test_dir = '/kaggle/input/face-mask-12k-images-dataset/Face Mask Dataset/Test'
val_dir = '/kaggle/input/face-mask-12k-images-dataset/Face Mask Dataset/Validation'

In [ ]:
train_datagen = ImageDataGenerator(horizontal_flip=True, zoom_range=0.2, shear_range=0.2, brightness_range=[0.6, 1.0], preprocessing_function=preprocess)
train_generator = train_datagen.flow_from_directory(directory=train_dir,
                                                    target_size=(input_dim, input_dim),
                                                    class_mode='categorical',
                                                    batch_size=batch_size)

val_datagen = ImageDataGenerator()
val_generator = train_datagen.flow_from_directory(directory=val_dir,
                                                  target_size=(input_dim, input_dim),
                                                  class_mode='categorical',
                                                  batch_size=batch_size)

test_datagen = ImageDataGenerator()
test_generator = train_datagen.flow_from_directory(directory=val_dir,
                                                   target_size=(input_dim, input_dim),
                                                   class_mode='categorical',
                                                   batch_size=batch_size)

In [ ]:
# fig, ax = plt.subplots(2, 3, figsize=(9, 7))
# ax = ax.flatten()

# img_batch = train_generator[0][0]
# label_batch = train_generator[0][1]
# for i, label, img in zip(np.arange(6), label_batch, img_batch):
#     ax[i].set_title(label)
#     ax[i].imshow(img / 255)
#     ax[i].axis('off')

# plt.show()

In [ ]:
history = model.fit(train_generator,
                    steps_per_epoch=len(train_generator),
                    epochs=3,
                    validation_data=val_generator,
                    validation_steps=len(val_generator))

In [ ]:
model.evaluate(test_generator)

In [ ]:
model.save('efficientnetv2_b3_syn6_orig3_softmax.h5')

In [ ]:
import json

with open('history_orig3_after_syn6.json', 'w') as f:
    json.dump(history.history, f, indent=4)